# 03 — Clustering per Zona

Fase 1 riset: "Mengelompokkan POI berdasarkan lokasi geografis ... untuk
memudahkan penjadwalan per hari (cluster per zona wilayah Jakarta)".
K-Means by lat/lon -> `zone_id`. Hasil jadi basis scope `time_matrix`
(04_time_matrix.ipynb) -- matrix in-zone dihitung antar-venue dalam zone
sama; all-pairs untuk lookup GA/PSO cross-zone.

In [ ]:
import os, sys
sys.path.insert(0, "..")
os.chdir(os.path.abspath(".."))  # jalankan dari root project

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import config

print("Root:", os.getcwd())

## [RUN] Jalankan clustering (K-Means k=8)

Input: `merged_venues_enriched.csv` (219 venue) → Output:
`jakarta_tourism_venues_clustered.csv` (+zone_id) & `jakarta_tourism_venues.csv`
(tanpa zone_id). Cache-aware: skip kalau output sudah ada (hapus file untuk rebuild).

In [ ]:
# [RUN] Clustering K-Means per zona (inline dari cluster_zones.py)
def run_clustering():
    df = pd.read_csv(config.MERGED_VENUES_ENRICHED_CSV)
    print(f"Venue input (merged): {len(df)}")

    coords = df[["latitude", "longitude"]].to_numpy()
    km = KMeans(n_clusters=config.CLUSTER_K, random_state=config.RANDOM_SEED, n_init=10)
    df["zone_id"] = km.fit_predict(coords)

    print(f"\nDistribusi venue per zone (k={config.CLUSTER_K}):")
    print(df["zone_id"].value_counts().sort_index().to_string())

    os.makedirs(os.path.dirname(config.CLUSTERED_VENUES_CSV), exist_ok=True)
    df.to_csv(config.CLUSTERED_VENUES_CSV, index=False)
    print(f"\nTersimpan -> {config.CLUSTERED_VENUES_CSV}")

    # Output pre-clustering (tanpa zone_id) untuk rekan/eksternal
    df.drop(columns=["zone_id"]).to_csv(config.TOURISM_VENUES_CSV, index=False)
    print(f"Tersimpan -> {config.TOURISM_VENUES_CSV}")
    return df


if os.path.exists(config.CLUSTERED_VENUES_CSV):
    print(f"[skip] {config.CLUSTERED_VENUES_CSV} sudah ada.")
    print("       Hapus file untuk rebuild clustering.")
else:
    run_clustering()

In [ ]:
df = pd.read_csv(config.CLUSTERED_VENUES_CSV)
print(f"Total venue: {len(df)}")
print(f"Jumlah zona (k={config.CLUSTER_K}):")
df["zone_id"].value_counts().sort_index()

## Sebaran geografis per zona

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
scatter = ax.scatter(df["longitude"], df["latitude"], c=df["zone_id"],
                      cmap="tab10", s=20, alpha=0.7)
ax.set_title(f"Cluster {len(df)} venue ke {config.CLUSTER_K} zona (K-Means)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.colorbar(scatter, label="zone_id", ticks=range(config.CLUSTER_K))
plt.show()

## Kategori dominan per zona

Sanity check: tiap zona harus punya karakter geografis yang masuk akal
(cth zona pesisir didominasi Beach/Theme Park, zona pusat kota didominasi
Museum/Monument).

In [ ]:
for zone in sorted(df["zone_id"].unique()):
    top_cat = df[df["zone_id"] == zone]["venue_category"].value_counts().head(3)
    n = len(df[df["zone_id"] == zone])
    print(f"zone {zone} ({n} venue): {dict(top_cat)}")

In [ ]:
pivot = pd.crosstab(df["zone_id"], df["venue_category"])
pivot.plot(kind="bar", stacked=True, figsize=(10, 6), legend=False)
plt.title("Distribusi kategori per zona")
plt.xlabel("zone_id")
plt.ylabel("jumlah venue")
plt.show()

## Silhouette Score — justifikasi pemilihan k

Evaluasi kualitas clustering (untuk laporan): silhouette mengukur seberapa
cocok venue dengan zonanya sendiri vs zona tetangga (−1..1, makin tinggi
makin baik). Dibandingkan k=4..12 sebagai justifikasi pemilihan k=8.

In [ ]:
# [STATS] Silhouette score utk k=4..12
from sklearn.metrics import silhouette_score

coords = df[["latitude", "longitude"]].to_numpy()
ks = range(4, 13)
scores = []
for k in ks:
    km_k = KMeans(n_clusters=k, random_state=config.RANDOM_SEED, n_init=10)
    labels = km_k.fit_predict(coords)
    scores.append(silhouette_score(coords, labels))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(ks), scores, marker="o", color="steelblue")
ax.axvline(config.CLUSTER_K, color="darkorange", linestyle="--",
           label=f"k={config.CLUSTER_K} (dipakai)")
ax.set_xlabel("Jumlah cluster (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette score K-Means per k — venue wisata Jakarta")
ax.legend()
plt.tight_layout()
plt.show()

for k, s in zip(ks, scores):
    mark = "  <- dipakai" if k == config.CLUSTER_K else ""
    print(f"k={k:2d}: {s:.4f}{mark}")